# 2. Цэвэрлэгээ ба боловсруулалт

Алхам бүрийг `src/clean.py` хийдэг; доорх лог нь `python -m src.clean`-ийн гаралт (бие даалтын Хэсэг 2).

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
P, R, F = ROOT / "data" / "processed", ROOT / "reports", ROOT / "figures"
def show(name):
    display(Image(filename=str(F / f"{name}.png"), width=760))

In [2]:
pd.read_csv(R / "cleaning_log.csv")

,step,description,rows_before,rows_after,note
0,C7,Давхардлыг хасав (game_id),252,252,NaN
1,C1,home_name: багийн нэрийг стандарт team_key руу...,252,252,16 түүхий нэр → 14 баг
2,C1,away_name: багийн нэрийг стандарт team_key руу...,252,252,16 түүхий нэр → 14 баг
3,C1b,Газрын нэрийг стандартчлав,252,252,6 хувилбар → 2
4,C11,Тоглолтын үе шатыг (regular/playoff) хуваарийн...,252,252,playoff: 37
5,C9,games: өгөгдлийн төрлийг тохируулав,252,252,NaN
6,C1,team_name: багийн нэрийг стандарт team_key руу...,504,504,16 түүхий нэр → 14 баг
7,C5,"'м-о' хэлбэрийн FG, TP, FT баганыг made/attemp...",504,504,NaN
8,C7,"Давхардлыг хасав (game_id, team_key)",504,504,NaN
9,C1,team_name: багийн нэрийг стандарт team_key руу...,318,318,16 түүхий нэр → 14 баг


## Жишээ: өмнө / дараа

In [3]:
raw = pd.read_csv(ROOT / "data" / "raw" / "players_season_raw.csv")
print("Багийн түүхий нэрс:", sorted(raw.team_name.unique()))
ps = pd.read_csv(P / "players_season.csv")
print("Стандарт team_key:", sorted(ps.team_key.unique()))

Багийн түүхий нэрс: ['BCH Knights', 'BISHRELT METAL', 'BUREN SCORE MONGOLIANS', 'Darkhan United', 'Khan Falcons', 'OMNI ERDENET MINERS', 'Omni Erdenet Miners', 'SELENGE BODONS', 'SG Apes', 'SUA TLG Tushig Bagana', 'Selenge Bodons', 'Sono Brothers', 'South Gobi Broncos', 'Western Warriors', 'XAC BRONCOS', 'ZAVKHAN BROTHERS']
Стандарт team_key: ['bch_knights', 'bishrelt_metal', 'burenscore_mongolians', 'darkhan_united', 'erdenet_miners', 'khan_falcons', 'khasin_khuleguud', 'selenge_bodons', 'sg_apes', 'sono_brothers', 'south_gobi_broncos', 'sua_tlg_tushig_bagana', 'western_warriors', 'zavkhan_brothers']


In [4]:
pg_raw = pd.read_csv(ROOT / "data" / "raw" / "player_games_raw.csv")
pg = pd.read_csv(P / "player_games.csv")
display(pg_raw[["name", "FG", "TP", "FT", "dnp"]].head(3)); display(pg[["player_id", "FGM", "FGA", "TPM", "TPA", "FTM", "FTA"]].head(3))

,name,FG,TP,FT,dnp
0,ALTANGEREL Azbayar,7-9,0-0,5-6,False
1,MILLER Ian,4-18,3-11,2-4,False
2,BAATAR-ERDENE Barsbold,5-13,1-8,2-2,False


,player_id,FGM,FGA,TPM,TPA,FTM,FTA
0,altangerel-azbayar,7,9,0,0,5,6
1,miller-ian,4,18,3,11,2,4
2,baatar-erdene-barsbold,5,13,1,8,2,2


In [5]:
prof = pd.read_csv(ROOT / "data" / "raw" / "players_profile_raw.csv")
prof[prof.birth_year.isin([1900, 2025])][["player_slug", "birth_year", "height_cm"]]

,player_slug,birth_year,height_cm
190,rodrigue-andela,1900.0,202.0
209,chinges-temuulen,2025.0,195.0
213,pok-simba,1900.0,210.0
218,manhom-lino,1900.0,NaN


## Дутуу утга, нэрийн тааруулалт, PPG шалгалт

In [6]:
display(pd.read_csv(R / "missing.csv"))
m = pd.read_csv(R / "name_matches.csv"); display(m[m.left_id != m.right_id].head(10))
v = pd.read_csv(R / "ppg_validation.csv"); v[v.flag]

,column,n_missing,pct_missing
0,position,0,0.00
1,height_cm,23,7.23
2,birth_year,15,4.72
3,FT_pct,38,11.95
4,TP_pct,34,10.69
5,FGA,0,0.00


,left_id,left_name,right_id,right_name,score,same_team,n_candidates
0,byron-wesley-2,BYRON Wesley,byron-wesley,BYRON Wesley,1.00,False,1
1,scrubb-jay,SCRUBB Jay,scrubb-jayden-amari,SCRUBB Jayden Amari,0.95,True,1
3,gankhuyag-dulguun-2,GANKHUYAG Dulguun,gankhuyag-dulguun,GANKHUYAG Dulguun,1.00,True,1
6,ochirbat-erdenetsogt-2,OCHIRBAT Erdenetsogt,ochirbat-erdenetsogt,OCHIRBAT Erdenetsogt,1.00,True,1
10,azzayaa-batkhuyag-2,AZZAYAA Batkhuyag,azzayaa-batkhuyag,AZZAYAA Batkhuyag,1.00,True,1
14,tolbert-jordan,TOLBERT Jordan,tolbert-jordan-wayne,TOLBERT Jordan Wayne,0.95,False,1
15,otgonbayar-usukhbayar-2,OTGONBAYAR Usukhbayar,otgonbayar-usukhbayar,OTGONBAYAR Usukhbayar,1.00,False,1
16,norvel-pelle,NORVEL Pelle,norvel-pelle-2,NORVEL Pelle,1.00,False,1
19,elbeg-tushig-2,ELBEG Tushig,elbeg-tushig,ELBEG Tushig,1.00,False,1
20,enkhtaivan-iderjavkhlan-3,ENKHTAIVAN Iderjavkhlan,enkhtaivan-iderjavkhlan-2,ENKHTAIVAN Iderjavkhlan,1.00,True,1


,player_id,season,team_key,PPG,GP,ppg_games,gp_games,diff,flag
31,azzayaa-batkhuyag,2025-2026,khan_falcons,13.0,27,14.25,24,1.25,True
44,davaadorj-dulguunmurun,2025-2026,bch_knights,10.4,18,4.12,8,6.28,True
62,tsogt-otgonjargal,2025-2026,khan_falcons,6.8,21,7.15,20,0.35,True
65,ishibashi-yuma,2025-2026,bch_knights,6.6,28,5.89,19,0.71,True
71,gantulga-tselmeg-2,2025-2026,bishrelt_metal,6.0,28,4.33,12,1.67,True
72,igari-wataru-2,2025-2026,burenscore_mongolians,5.9,21,6.75,16,0.85,True
121,erdenesaikhan-purevdorj,2025-2026,khasin_khuleguud,2.5,26,2.10,21,0.40,True
137,balt-barsbold,2025-2026,darkhan_united,1.6,14,2.00,9,0.40,True


## Шинэ хувьсагчид

`age`, `pos_group`, per-36 (`pts36` …), `ts_pct`, `tpa_rate`, `ftr`, `scorer_tier`, `is_national`, `region`; багийн тоглолтод Four Factors (`efg`, `tov_pct`, `orb_pct`, `ftr`), `ortg`, `drtg`, `net`; тоглолтод Elo.

In [7]:
ps[["player_name", "season", "PPG", "pts36", "ts_pct", "tpa_rate", "scorer_tier", "pos_group", "age", "region", "is_national"]].head()

,player_name,season,PPG,pts36,ts_pct,tpa_rate,scorer_tier,pos_group,age,region,is_national
0,BYRON Wesley,2025-2026,27.9,30.904615,0.587171,0.262443,Од,Forward,33.0,Улаанбаатар,False
1,UORREN Davion,2025-2026,26.7,31.006452,0.582254,0.263485,Од,Forward,26.0,Улаанбаатар,False
2,SCRUBB Jay,2025-2026,25.2,27.913846,0.579877,0.490566,Од,Guard,25.0,Орон нутаг,False
3,MCKENZIE Moore,2025-2026,24.3,24.367688,0.519431,0.317269,Од,Forward,33.0,Орон нутаг,False
4,BRANDIS Raley Ross,2025-2026,22.3,26.582781,0.668203,0.453782,Од,Guard,38.0,Орон нутаг,False


**[Ноорог — өөрийн үгээр дахин бичих]** MBA-ийн улирлын хүснэгт зөвхөн энгийн улирлыг тооцдог нь C10 шалгалтаар илэрсэн; 1900 гэх мэт placeholder төрсөн оныг C12 алхамд засав; багийн нэрийн 16 хувилбарыг 14 баг болгов.